In [3]:
!nvidia-smi

Tue Nov  3 11:28:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install -q ultralytics fiftyone

## Connect Google Drive (to save your work)

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = "/content/drive/MyDrive/object_tracking"
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
print("Working in:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/object_tracking


Cell 4: Download the Open Images subset

In [6]:
import fiftyone as fo
import fiftyone.zoo as foz

CLASSES = ["Person", "Bottle", "Mug", "Mobile phone", "Laptop",
           "Backpack", "Headphones", "Computer mouse", "Book", "Remote control"]

for split, n in [("train", 3000), ("validation", 400)]:
    ds = foz.load_zoo_dataset(
        "open-images-v7", split=split,
        label_types=["detections"], classes=CLASSES,
        max_samples=n, dataset_name=f"oi_{split}", shuffle=True, seed=0)
    ds.export(export_dir=f"/content/oi_yolo/{split}",
              dataset_type=fo.types.YOLOv5Dataset,
              label_field="ground_truth", classes=CLASSES, split="val" if split=="validation" else "train")
print("done")

INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/open-images-v7/train' if necessary


Necessary images already downloaded


INFO:fiftyone.utils.openimages:Necessary images already downloaded


Existing download of split 'train' is sufficient


INFO:fiftyone.zoo.datasets:Existing download of split 'train' is sufficient


Loading existing dataset 'oi_train'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


... (206 outputs left out)

/usr/local/lib/python3.13/dist-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'Baseball glove' not in provided classes
  warnings.warn(msg)
/usr/local/lib/python3.13/dist-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'Baseball bat' not in provided classes
  warnings.warn(msg)
/usr/local/lib/python3.13/dist-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'Picture frame' not in provided classes
  warnings.warn(msg)
/usr/local/lib/python3.13/dist-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'Curtain' not in provided classes
  warnings.warn(msg)
/usr/local/lib/python3.13/dist-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'Cat' not in provided classes
  warnings.warn(msg)
/usr/local/lib/python3.13/dist-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'Plumbing fixture' not in provided classes
  warnings.war

 100% |█████████████████| 400/400 [4.6s elapsed, 0s remaining, 113.6 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 400/400 [4.6s elapsed, 0s remaining, 113.6 samples/s]      


done


Cell 5: Merge your AirPods data

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = "/content/drive/MyDrive/object_tracking"
os.makedirs(DRIVE, exist_ok=True)

CLASSES = ["Person", "Bottle", "Mug", "Mobile phone", "Laptop",
           "Backpack", "Headphones", "Computer mouse", "Book", "Remote control"]

print("DRIVE:", DRIVE)
print("dataset.zip found:", os.path.exists(f"{DRIVE}/dataset.zip"))
print("Open Images export found:", os.path.exists("/content/oi_yolo/train"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE: /content/drive/MyDrive/object_tracking
dataset.zip found: False
Open Images export found: True


In [8]:
import zipfile, re, shutil
from pathlib import Path

zipfile.ZipFile(f"{DRIVE}/airpods_tracking_v2.v1-airpods_yolo11_v1.yolov11.zip").extractall("/content/export_all")

SRC, OUT = Path("/content/export_all"), Path("/content/mine/dataset")
shutil.rmtree(OUT, ignore_errors=True)
split_of = {"AirPods_1": "train", "AirPods_2": "train", "AirPods_3": "val", "AirPods_4": "test"}

for img in SRC.rglob("*.jpg"):
    m = re.match(r"(AirPods_\d+)", img.name)
    if not m: continue
    s = split_of[m.group(1)]
    lbl = img.parent.parent / "labels" / (img.stem + ".txt")
    (OUT / s / "images").mkdir(parents=True, exist_ok=True)
    (OUT / s / "labels").mkdir(parents=True, exist_ok=True)
    shutil.copy(img, OUT / s / "images" / img.name)
    shutil.copy(lbl, OUT / s / "labels" / lbl.name)

for s in ["train", "val", "test"]:
    print(s, len(list((OUT / s / "images").glob("*.jpg"))))

train 129
val 62
test 61


In [9]:
for f in Path("/content/mine/dataset").rglob("labels/*.txt"):
    out = []
    for ln in f.read_text().strip().splitlines():
        p = ln.split()
        if len(p) != 5:
            xs, ys = list(map(float, p[1::2])), list(map(float, p[2::2]))
            x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
            ln = f"{p[0]} {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}"
        out.append(ln)
    f.write_text("\n".join(out) + "\n")
print("labels cleaned")

labels cleaned


In [10]:
import shutil
from pathlib import Path

mine = Path("/content/mine/dataset")
AIR_ID = len(CLASSES)                      # AirPods = last class
OUT = Path("/content/merged")
shutil.rmtree(OUT, ignore_errors=True)

# Open Images export
for s, oi in [("train", "train"), ("val", "validation")]:
    for kind in ["images", "labels"]:
        src = Path(f"/content/oi_yolo/{oi}/{kind}/{s}")
        dst = OUT / s / kind; dst.mkdir(parents=True, exist_ok=True)
        if not src.exists():
            print("MISSING:", src); continue
        for f in src.glob("*"): shutil.copy(f, dst / f.name)

# AirPods (train x5 to balance, val x1)
for s, reps in [("train", 5), ("val", 1)]:
    for img in (mine / s / "images").glob("*.jpg"):
        lbl = mine / s / "labels" / (img.stem + ".txt")
        txt = "\n".join(f"{AIR_ID} " + " ".join(l.split()[1:])
                        for l in lbl.read_text().splitlines() if l.strip())
        for r in range(reps):
            shutil.copy(img, OUT / s / "images" / f"air{r}_{img.name}")
            (OUT / s / "labels" / f"air{r}_{img.stem}.txt").write_text(txt)

for s in ["train", "val"]:
    print(s, "images:", len(list((OUT / s / "images").glob("*"))),
          "labels:", len(list((OUT / s / "labels").glob("*.txt"))))


train images: 3645 labels: 3645
val images: 462 labels: 462


Cell 6: data.yaml

In [11]:
import yaml
names = CLASSES + ["AirPods"]
yaml.safe_dump({"path": "/content/merged", "train": "train/images", "val": "val/images",
                "names": {i: n for i, n in enumerate(names)}},
               open("/content/merged/data.yaml", "w"))
print(names)

['Person', 'Bottle', 'Mug', 'Mobile phone', 'Laptop', 'Backpack', 'Headphones', 'Computer mouse', 'Book', 'Remote control', 'AirPods']


Cell 7: Train

In [12]:
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.train(data="/content/merged/data.yaml", epochs=40, imgsz=640, batch=32,
            patience=0, project=f"{DRIVE}/runs", name="multi_v1",
            exist_ok=True, workers=2)

Ultralytics 8.4.200 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=multi_v1, nbs=64, nms=None, opset=None, optimize=False, optimizer

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  8, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bcf7991a190>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046, 

Cell 8: Honest tests

In [13]:
best = YOLO(f"{DRIVE}/runs/multi_v1/weights/best.pt")
best.val()   # prints AP per class

Ultralytics 8.4.200 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,417,057 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1672.7±893.0 MB/s, size: 207.0 KB)
val: Scanning /content/merged/val/labels.cache... 462 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 462/462 114.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.9it/s 10.0s
                   all        462        974      0.597      0.445      0.437      0.361
                Person        373        866      0.364      0.401      0.277      0.158
                Bottle         10         14      0.588      0.511      0.555      0.391
                   Mug          2          2      0.722        0.5      0.513      0.462
          Mobile phone          7         10      0.585        0.8      0.762      0.661
                Laptop          4          4

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  8, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bcf436b49f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046, 

In [17]:
from ultralytics import YOLO, YOLOWorld

coco  = YOLO("yolo11s.pt")
air = YOLO("/content/drive/MyDrive/object_tracking/runs/multi_v1/weights/best.pt")
world = None
COCO_NAMES = {v.lower(): k for k, v in coco.names.items()}

def get_model(target):
    global world
    t = target.lower().strip()
    if t in ("airpods", "airpod", "airbuds", "earbuds"):
        return air, None, 0.5
    if t in COCO_NAMES:
        return coco, [COCO_NAMES[t]], 0.4        # filter to that class id
    if world is None:
        world = YOLOWorld("yolov8s-worldv2.pt")
    world.set_classes([t])
    return world, None, 0.25
print("COCO classes:", len(coco.names))
print("AirPods model classes:", air.names)
model, classes, conf = get_model("bottle")
print("bottle ->", type(model).__name__, classes, conf)
# use: model, classes, conf = get_model("bottle")
# model.track(frame, persist=True, classes=classes, conf=conf, tracker="bytetrack.yaml")

COCO classes: 80
AirPods model classes: {0: 'Person', 1: 'Bottle', 2: 'Mug', 3: 'Mobile phone', 4: 'Laptop', 5: 'Backpack', 6: 'Headphones', 7: 'Computer mouse', 8: 'Book', 9: 'Remote control', 10: 'AirPods'}
bottle -> YOLO [39] 0.4


In [18]:
from google.colab import files
files.download("/content/drive/MyDrive/object_tracking/runs/multi_v1/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>